# E-commerce A/B Testing with Revenue Forecasting

## Notebook 1: Data Pull and Validation

This notebook loads the experiment dataset and the store sales dataset, checks data quality, standardizes fields, and saves clean raw files for the rest of the project.

## Goals

1. Load the A/B testing dataset
2. Load the store sales forecasting dataset
3. Validate schema and missingness
4. Parse dates and standardize columns
5. Save cleaned raw outputs

Links to datasets:

[A/B Test](https://www.kaggle.com/datasets/amirmotefaker/ab-testing-dataset)

[Time Series](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)

In [17]:
# Import libraries

from pathlib import Path
import pandas as pd
import numpy as np

In [39]:
# Set display options and paths

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "01-raw"
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "02-preprocessed"

AB_TEST_DIR = RAW_DIR / "ab_test"
STORE_SALES_DIR = RAW_DIR / "store_sales"

print("Project root:", PROJECT_ROOT)
print("A/B test dir exists:", AB_TEST_DIR.exists())
print("Store sales dir exists:", STORE_SALES_DIR.exists())

Project root: c:\Users\tevin\OneDrive\Desktop\General-Machine-Learning\revenue_forecasting
A/B test dir exists: True
Store sales dir exists: True


In [19]:
# List raw files

print("A/B test files:")
for file in AB_TEST_DIR.iterdir():
    print(file.name)

print("\nStore sales files:")
for file in STORE_SALES_DIR.iterdir():
    print(file.name)

A/B test files:
control_group.csv
test_group.csv

Store sales files:
holidays_events.csv
oil.csv
sample_submission.csv
stores.csv
test.csv
train.csv
transactions.csv


## Load A/B Testing Data

This section loads the control and test groups, labels the variants, and combines them into one experiment dataset.

In [20]:
# Load A/B test files

control_df = pd.read_csv(AB_TEST_DIR / "control_group.csv", sep=";")
test_df = pd.read_csv(AB_TEST_DIR / "test_group.csv", sep=";")

control_df["variant"] = "control"
test_df["variant"] = "test"

ab_df = pd.concat([control_df, test_df], ignore_index=True)

print("Control shape:", control_df.shape)
print("Test shape:", test_df.shape)
print("Combined A/B shape:", ab_df.shape)

ab_df.head()

Control shape: (30, 11)
Test shape: (30, 11)
Combined A/B shape: (60, 11)


,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase,variant
0,Control Campaign,1.08.2019,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0,control
1,Control Campaign,2.08.2019,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0,control
2,Control Campaign,3.08.2019,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0,control
3,Control Campaign,4.08.2019,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0,control
4,Control Campaign,5.08.2019,1835,NaN,NaN,NaN,NaN,NaN,NaN,NaN,control


## A/B Test Schema Check

This section inspects column names, types, and null values in the experiment dataset.

In [21]:
# Inspect A/B dataset schema

print(ab_df.columns.tolist())
print("\n")
print(ab_df.info())
print("\nMissing values:")
print(ab_df.isna().sum())

['Campaign Name', 'Date', 'Spend [USD]', '# of Impressions', 'Reach', '# of Website Clicks', '# of Searches', '# of View Content', '# of Add to Cart', '# of Purchase', 'variant']


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Campaign Name        60 non-null     object 
 1   Date                 60 non-null     object 
 2   Spend [USD]          60 non-null     int64  
 3   # of Impressions     59 non-null     float64
 4   Reach                59 non-null     float64
 5   # of Website Clicks  59 non-null     float64
 6   # of Searches        59 non-null     float64
 7   # of View Content    59 non-null     float64
 8   # of Add to Cart     59 non-null     float64
 9   # of Purchase        59 non-null     float64
 10  variant              60 non-null     object 
dtypes: float64(7), int64(1), object(3)
memory usage: 5.3+ KB
None

## Standardize A/B Test Columns

This section renames the experiment columns into consistent snake_case names for downstream analysis.

In [22]:
# Standardize A/B test column names

ab_df = ab_df.rename(columns={
    "Campaign Name": "campaign_name",
    "Date": "date",
    "Spend [USD]": "spend_usd",
    "# of Impressions": "impressions",
    "Reach": "reach",
    "# of Website Clicks": "website_clicks",
    "# of Searches": "searches",
    "# of View Content": "view_content",
    "# of Add to Cart": "add_to_cart",
    "# of Purchase": "purchase"
})

ab_df.columns = [col.strip().lower().replace(" ", "_") for col in ab_df.columns]

ab_df.head()

,campaign_name,date,spend_usd,impressions,reach,website_clicks,searches,view_content,add_to_cart,purchase,variant
0,Control Campaign,1.08.2019,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0,control
1,Control Campaign,2.08.2019,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0,control
2,Control Campaign,3.08.2019,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0,control
3,Control Campaign,4.08.2019,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0,control
4,Control Campaign,5.08.2019,1835,NaN,NaN,NaN,NaN,NaN,NaN,NaN,control


## A/B Test Validation

This section checks date coverage, missing values, and variant-level record counts.

In [30]:
# Summarize A/B test data quality

ab_summary = pd.DataFrame({
    "metric": [
        "row_count",
        "n_variants",
        "min_date",
        "max_date"
    ],
    "value": [
        len(ab_df),
        ab_df["variant"].nunique(),
        ab_df["date"].min(),
        ab_df["date"].max()
    ]
})
variant_summary = ab_df.groupby("variant").agg(
    rows=("variant", "count"),
    total_spend=("spend_usd", "sum"),
    total_impressions=("impressions", "sum"),
    total_clicks=("website_clicks", "sum"),
    total_purchases=("purchase", "sum")
).reset_index()

ab_summary, variant_summary

(       metric      value
 0   row_count         60
 1  n_variants          2
 2    min_date  1.08.2019
 3    max_date  9.08.2019,
    variant  rows  total_spend  total_impressions  total_clicks  total_purchases
 0  control    30        68653          3177233.0      154303.0          15161.0
 1     test    30        76892          2237544.0      180970.0          15637.0)

## Load Store Sales Data

This section loads the core forecasting tables and supporting lookup tables used later in the project.

In [25]:
# Load store sales files

train_df = pd.read_csv(STORE_SALES_DIR / "train.csv")
test_df_store = pd.read_csv(STORE_SALES_DIR / "test.csv")
transactions_df = pd.read_csv(STORE_SALES_DIR / "transactions.csv")
oil_df = pd.read_csv(STORE_SALES_DIR / "oil.csv")
holidays_df = pd.read_csv(STORE_SALES_DIR / "holidays_events.csv")
stores_df = pd.read_csv(STORE_SALES_DIR / "stores.csv")

print("train_df:", train_df.shape)
print("test_df_store:", test_df_store.shape)
print("transactions_df:", transactions_df.shape)
print("oil_df:", oil_df.shape)
print("holidays_df:", holidays_df.shape)
print("stores_df:", stores_df.shape)

train_df: (3000888, 6)
test_df_store: (28512, 5)
transactions_df: (83488, 3)
oil_df: (1218, 2)
holidays_df: (350, 6)
stores_df: (54, 5)


## Store Sales Schema Check

This section reviews the structure of the forecasting tables before type conversion and validation.

In [26]:
# Inspect train dataset schema

print(train_df.columns.tolist())
print("\n")
print(train_df.info())
print("\nMissing values:")
print(train_df.isna().sum())

['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB
None

Missing values:
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64


In [31]:
# Inspect supporting datasets

print("Transactions info")
print(transactions_df.info())
print(transactions_df.isna().sum())

print("\nOil info")
print(oil_df.info())
print(oil_df.isna().sum())

print("\nHolidays info")
print(holidays_df.info())
print(holidays_df.isna().sum())

print("\nStores info")
print(stores_df.info())
print(stores_df.isna().sum())

Transactions info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          83488 non-null  object
 1   store_nbr     83488 non-null  int64 
 2   transactions  83488 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.9+ MB
None
date            0
store_nbr       0
transactions    0
dtype: int64

Oil info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1218 non-null   object 
 1   dcoilwtico  1175 non-null   float64
dtypes: float64(1), object(1)
memory usage: 19.2+ KB
None
date           0
dcoilwtico    43
dtype: int64

Holidays info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtyp

### Parse Date Columns

This section standardizes all date columns across the A/B and forecasting datasets.

In [32]:
# Parse date columns

ab_df["date"] = pd.to_datetime(ab_df["date"], errors="coerce")
train_df["date"] = pd.to_datetime(train_df["date"], errors="coerce")
test_df_store["date"] = pd.to_datetime(test_df_store["date"], errors="coerce")
transactions_df["date"] = pd.to_datetime(transactions_df["date"], errors="coerce")
oil_df["date"] = pd.to_datetime(oil_df["date"], errors="coerce")
holidays_df["date"] = pd.to_datetime(holidays_df["date"], errors="coerce")

In [33]:
# Parse numeric columns

train_df["sales"] = pd.to_numeric(train_df["sales"], errors="coerce")
train_df["onpromotion"] = pd.to_numeric(train_df["onpromotion"], errors="coerce")

test_df_store["onpromotion"] = pd.to_numeric(test_df_store["onpromotion"], errors="coerce")

transactions_df["transactions"] = pd.to_numeric(transactions_df["transactions"], errors="coerce")
oil_df["dcoilwtico"] = pd.to_numeric(oil_df["dcoilwtico"], errors="coerce")

## Forecasting Target Validation

This section checks the main sales target for date coverage, store coverage, family coverage, and missing values.

In [35]:
# Summarize train dataset coverage

train_summary = pd.DataFrame({
    "metric": [
        "row_count",
        "min_date",
        "max_date",
        "n_stores",
        "n_families",
        "missing_sales",
        "missing_onpromotion"
    ],
    "value": [
        len(train_df),
        train_df["date"].min(),
        train_df["date"].max(),
        train_df["store_nbr"].nunique(),
        train_df["family"].nunique(),
        train_df["sales"].isna().sum(),
        train_df["onpromotion"].isna().sum()
    ]
})

train_summary

,metric,value
0,row_count,3000888
1,min_date,2013-01-01 00:00:00
2,max_date,2017-08-15 00:00:00
3,n_stores,54
4,n_families,33
5,missing_sales,0
6,missing_onpromotion,0


## Oil Series Validation

This section checks oil price coverage and missingness before the series is used in downstream analysis.

In [36]:
# Summarize oil series quality

oil_summary = pd.DataFrame({
    "metric": [
        "row_count",
        "min_date",
        "max_date",
        "missing_oil_values"
    ],
    "value": [
        len(oil_df),
        oil_df["date"].min(),
        oil_df["date"].max(),
        oil_df["dcoilwtico"].isna().sum()
    ]
})

oil_summary

,metric,value
0,row_count,1218
1,min_date,2013-01-01 00:00:00
2,max_date,2017-08-31 00:00:00
3,missing_oil_values,43


## Save Clean Raw Files

This section writes the validated datasets back to the raw data folder for reuse in later notebooks.

In [40]:
# Save preprocessed datasets

ab_df.to_csv(PREPROCESSED_DIR / "ab_test_clean.csv", index=False)

train_df.to_csv(PREPROCESSED_DIR / "store_sales_train_clean.csv", index=False)
test_df_store.to_csv(PREPROCESSED_DIR / "store_sales_test_clean.csv", index=False)
transactions_df.to_csv(PREPROCESSED_DIR / "store_sales_transactions_clean.csv", index=False)
oil_df.to_csv(PREPROCESSED_DIR / "store_sales_oil_clean.csv", index=False)
holidays_df.to_csv(PREPROCESSED_DIR / "store_sales_holidays_clean.csv", index=False)
stores_df.to_csv(PREPROCESSED_DIR / "store_sales_stores_clean.csv", index=False)

print("Preprocessed files saved successfully.")

Preprocessed files saved successfully.
